# 📊 Análise Estatística — Zika Vírus (SINAN 2018–2026)

**Disciplina**: Banco de Dados / Modelagem Estatística  
**Dataset**: `ZIKA_BR_2018_2026_UNIFICADO.csv` — 236.398 notificações  
**Fonte**: SINAN — Sistema de Informação de Agravos de Notificação (DataSUS/MS)  

---

### Entregas deste notebook:
1. **Sazonalidade** — Decomposição da série temporal e identificação de padrões cíclicos
2. **Tendência por UF** — Evolução temporal dos casos confirmados por estado
3. **Previsão de Casos (Prophet)** — Modelagem preditiva com validação cruzada
4. **Agrupamento de Municípios (K-Means)** — Clustering por perfil epidemiológico

---
## ⚙️ Etapa 0 — Setup e Conexão com o Banco

Configuração de bibliotecas, conexão com PostgreSQL (NeonDB) e helpers.

In [ ]:
# ── Instalação de dependências (executar apenas 1 vez) ──
# !pip install prophet statsmodels scikit-learn sqlalchemy psycopg2-binary python-dotenv seaborn matplotlib pandas

In [ ]:
# ── Imports ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
from datetime import datetime

# Conexão com o banco
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

warnings.filterwarnings('ignore')

# ── Conexão ──
load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')  # string de conexão NeonDB no .env
engine = create_engine(DATABASE_URL)

def query_df(sql):
    """Executa SQL e retorna DataFrame."""
    return pd.read_sql(sql, engine)

print('✅ Conexão configurada com sucesso.')

In [ ]:
# ── Estilo global dos gráficos ──
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 10,
    'figure.dpi': 150
})
sns.set_style('whitegrid')

# Paleta personalizada para o projeto
COR_PRINCIPAL  = '#E63946'  # vermelho epidemiológico
COR_SECUNDARIA = '#457B9D'  # azul complementar
COR_DESTAQUE   = '#F4A261'  # laranja para destaques
PALETA_UFS     = sns.color_palette('tab10', 10)

print('✅ Estilo de gráficos configurado.')

---
## 📈 Etapa 1 — Análise de Sazonalidade

Objetivo: identificar padrões cíclicos (sazonais) na incidência de Zika,
 utilizando a `vw_serie_temporal_semanal` que agrega casos por semana epidemiológica.

**Hipótese epidemiológica**: espera-se pico de casos entre dezembro e março
 (verão + período chuvoso → proliferação do Aedes aegypti).

### 1.1 — Extração e preparação dos dados

In [ ]:
# ── Query na view já criada no banco ──
df_semanal = query_df('SELECT * FROM vw_serie_temporal_semanal;')

print(f'Registros carregados: {len(df_semanal)}')
print(f'Colunas: {list(df_semanal.columns)}')
print(f'Período: {df_semanal["nu_ano"].min()} a {df_semanal["nu_ano"].max()}')
df_semanal.head(10)

In [ ]:
# ── Converter ano + semana epidemiológica → data real ──
# Formato ISO: %G = ano ISO, %V = semana ISO, %u = dia da semana (1=seg)
df_semanal['ds'] = pd.to_datetime(
    df_semanal['nu_ano'].astype(str) + '-W' +
    df_semanal['sem_pri'].astype(str).str.zfill(2) + '-1',
    format='%G-W%V-%u',
    errors='coerce'
)

df_semanal = df_semanal.dropna(subset=['ds']).sort_values('ds').reset_index(drop=True)
print(f'Registros após conversão: {len(df_semanal)}')
print(f'Intervalo: {df_semanal["ds"].min().date()} → {df_semanal["ds"].max().date()}')

### 1.2 — Série temporal completa (notificações × confirmados)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(df_semanal['ds'], df_semanal['total_casos'],
        label='Total de notificações', color=COR_SECUNDARIA, alpha=0.6, linewidth=1)
ax.plot(df_semanal['ds'], df_semanal['casos_confirmados'],
        label='Casos confirmados', color=COR_PRINCIPAL, alpha=0.9, linewidth=1.2)

ax.set_title('Série Temporal Semanal — Zika Vírus (SINAN 2018–2026)')
ax.set_xlabel('Data (semana epidemiológica)')
ax.set_ylabel('Nº de Casos')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('serie_temporal_semanal.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: serie_temporal_semanal.png')

### 1.3 — Decomposição sazonal (Tendência + Sazonalidade + Resíduo)

Utiliza decomposição aditiva com período de 52 semanas (1 ano) para separar
 os 3 componentes da série.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Cria série com frequência semanal regular (preenche gaps com 0)
ts = df_semanal.set_index('ds')['total_casos'].asfreq('W-MON', fill_value=0)

decomposicao = seasonal_decompose(ts, model='additive', period=52)

fig = decomposicao.plot()
fig.set_size_inches(16, 10)
fig.suptitle('Decomposição da Série Temporal — Zika Vírus (Sazonalidade Anual)',
             fontsize=14, y=1.02)

plt.tight_layout()
plt.savefig('decomposicao_sazonal.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: decomposicao_sazonal.png')

**Interpretação:**
- **Trend (Tendência)**: direção geral da série — Zika em alta ou queda?
- **Seasonal (Sazonalidade)**: padrão que se repete anualmente — pico no verão?
- **Residual (Resíduo)**: variação não explicada — surtos atípicos, outliers.

### 1.4 — Boxplot de sazonalidade mensal

In [ ]:
df_semanal['mes'] = df_semanal['ds'].dt.month
df_semanal['nome_mes'] = df_semanal['ds'].dt.strftime('%b')  # Jan, Feb...

# Ordenação correta dos meses
ordem_meses = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=df_semanal, x='nome_mes', y='total_casos',
            order=ordem_meses, palette='YlOrRd', ax=ax)

ax.set_title('Distribuição de Casos Semanais por Mês — Sazonalidade do Zika')
ax.set_xlabel('Mês')
ax.set_ylabel('Total de Casos (por semana)')

plt.tight_layout()
plt.savefig('sazonalidade_mensal.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: sazonalidade_mensal.png')

### 1.5 — Heatmap: Semana Epidemiológica × Ano

Permite visualizar em qual semana e ano ocorreram os maiores surtos.

In [ ]:
pivot_sem = df_semanal.pivot_table(
    index='sem_pri', columns='nu_ano',
    values='total_casos', fill_value=0
)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot_sem, cmap='YlOrRd', linewidths=0.1, ax=ax)
ax.set_title('Heatmap — Casos por Semana Epidemiológica e Ano')
ax.set_xlabel('Ano')
ax.set_ylabel('Semana Epidemiológica')

plt.tight_layout()
plt.savefig('heatmap_semana_ano.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: heatmap_semana_ano.png')

---
## 🗺️ Etapa 2 — Tendência por UF

Objetivo: identificar quais Unidades Federativas concentram mais casos e
 como a incidência evoluiu ao longo dos anos.

### 2.1 — Extração dos dados

In [ ]:
df_uf = query_df('SELECT * FROM vw_casos_uf_ano;')
# Colunas: sg_uf | nu_ano | total_notificacoes | casos_confirmados

print(f'Registros: {len(df_uf)}')
print(f'UFs presentes: {df_uf["sg_uf"].nunique()}')
df_uf.head(10)

### 2.2 — Heatmap: UF × Ano (Casos Confirmados)

In [ ]:
pivot_uf = df_uf.pivot_table(
    index='sg_uf', columns='nu_ano',
    values='casos_confirmados', fill_value=0
)

# Ordena UFs pelo total acumulado (decrescente)
ordem_uf = pivot_uf.sum(axis=1).sort_values(ascending=False).index
pivot_uf = pivot_uf.loc[ordem_uf]

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot_uf, annot=True, fmt='g', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title('Casos Confirmados de Zika por UF e Ano')
ax.set_xlabel('Ano')
ax.set_ylabel('UF de Residência (ordenada por total)')

plt.tight_layout()
plt.savefig('heatmap_uf_ano.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: heatmap_uf_ano.png')

### 2.3 — Evolução temporal: Top 5 UFs

In [ ]:
# Top 5 UFs com mais casos confirmados acumulados
top5_ufs = (df_uf.groupby('sg_uf')['casos_confirmados']
            .sum().nlargest(5).index.tolist())
print(f'Top 5 UFs: {top5_ufs}')

fig, ax = plt.subplots(figsize=(14, 6))
for i, uf in enumerate(top5_ufs):
    dados_uf = df_uf[df_uf['sg_uf'] == uf].sort_values('nu_ano')
    ax.plot(dados_uf['nu_ano'], dados_uf['casos_confirmados'],
            marker='o', label=uf, color=PALETA_UFS[i], linewidth=2)

ax.set_title('Evolução de Casos Confirmados — Top 5 UFs')
ax.set_xlabel('Ano')
ax.set_ylabel('Casos Confirmados')
ax.legend(title='UF')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('tendencia_top5_uf.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: tendencia_top5_uf.png')

### 2.4 — Ranking acumulado por UF (barras horizontais)

In [ ]:
ranking = (df_uf.groupby('sg_uf')[['total_notificacoes', 'casos_confirmados']]
           .sum().sort_values('casos_confirmados', ascending=True))

fig, ax = plt.subplots(figsize=(10, 10))
ranking.plot.barh(ax=ax, color=[COR_SECUNDARIA, COR_PRINCIPAL])
ax.set_title('Total Acumulado (2018–2026) por UF')
ax.set_xlabel('Nº de Casos')
ax.set_ylabel('UF')
ax.legend(['Notificações', 'Confirmados'])

plt.tight_layout()
plt.savefig('ranking_uf_acumulado.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: ranking_uf_acumulado.png')

---
## 🔮 Etapa 3 — Previsão de Casos com Prophet

O [Prophet](https://facebook.github.io/prophet/) é um modelo aditivo de séries temporais
 desenvolvido pelo Meta, ideal para dados com **sazonalidade forte** e **gaps**.

Modelo: $y(t) = g(t) + s(t) + h(t) + \epsilon_t$
- $g(t)$: tendência (linear ou logística)
- $s(t)$: sazonalidade (Fourier)
- $h(t)$: efeitos de feriados/eventos
- $\epsilon_t$: ruído

### 3.1 — Preparação dos dados para o Prophet

In [ ]:
# Prophet exige exatamente 2 colunas: ds (datetime) e y (numérico)
df_prophet = df_semanal[['ds', 'total_casos']].rename(
    columns={'total_casos': 'y'}
).copy()

df_prophet = df_prophet.sort_values('ds').reset_index(drop=True)

print(f'Registros para treino: {len(df_prophet)}')
print(f'Período: {df_prophet["ds"].min().date()} → {df_prophet["ds"].max().date()}')
df_prophet.tail()

### 3.2 — Treinamento do modelo

In [ ]:
from prophet import Prophet

modelo = Prophet(
    seasonality_mode='additive',    # sazonalidade aditiva (padrão epidemiológico)
    yearly_seasonality=True,         # captura ciclo anual
    weekly_seasonality=False,        # dados semanais, não diários
    daily_seasonality=False,
    changepoint_prior_scale=0.05,    # controla flexibilidade da tendência
    interval_width=0.95              # intervalo de confiança de 95%
)

modelo.fit(df_prophet)
print('✅ Modelo treinado com sucesso.')

### 3.3 — Previsão (52 semanas à frente)

In [ ]:
# Gera dataframe com datas futuras (1 ano = 52 semanas)
futuro = modelo.make_future_dataframe(periods=52, freq='W')
previsao = modelo.predict(futuro)

print(f'Previsão gerada até: {previsao["ds"].max().date()}')
print(f'\nÚltimas 5 semanas previstas:')
previsao[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

In [ ]:
# ── Gráfico principal: histórico + previsão ──
fig1 = modelo.plot(previsao)
fig1.set_size_inches(16, 6)
plt.title('Previsão de Casos de Zika — Prophet (52 semanas à frente)', fontsize=14)
plt.xlabel('Data')
plt.ylabel('Nº de Casos')

plt.tight_layout()
plt.savefig('prophet_previsao.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: prophet_previsao.png')

In [ ]:
# ── Componentes (tendência + sazonalidade anual) ──
fig2 = modelo.plot_components(previsao)
fig2.set_size_inches(14, 8)

plt.tight_layout()
plt.savefig('prophet_componentes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: prophet_componentes.png')

### 3.4 — Validação cruzada (Cross-Validation)

Avalia a qualidade preditiva do modelo usando janelas deslizantes:
- **initial**: período mínimo de treino (730 dias = 2 anos)
- **period**: a cada 30 dias faz novo corte
- **horizon**: prevê 90 dias à frente em cada corte

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

df_cv = cross_validation(
    modelo,
    horizon='90 days',
    period='30 days',
    initial='730 days'
)

df_metricas = performance_metrics(df_cv)

print('=== Métricas de Validação Cruzada ===')
print(df_metricas[['horizon', 'rmse', 'mae', 'mape']].to_string(index=False))

In [ ]:
# ── Gráfico de MAPE por horizonte ──
from prophet.plot import plot_cross_validation_metric

fig3 = plot_cross_validation_metric(df_cv, metric='mape')
fig3.set_size_inches(12, 5)
plt.title('MAPE por Horizonte de Previsão — Validação Cruzada')

plt.tight_layout()
plt.savefig('prophet_cv_mape.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: prophet_cv_mape.png')

### 3.5 — (Bônus) Prophet por UF — Top 5 estados

Gera previsões individualizadas para os 5 estados com maior incidência.

In [ ]:
# ── Query: série semanal por UF (não existe view pronta para isso) ──
SQL_UF_SEMANAL = """
SELECT
    u.sg_uf,
    n.nu_ano,
    c.sem_pri,
    COUNT(*) AS total_casos
FROM tb_notificacao n
JOIN tb_dados_clinicos c           ON n.id_notificacao = c.id_notificacao
JOIN tb_geografia_epidemiologica g ON n.id_notificacao = g.id_notificacao
JOIN tb_municipio m                ON g.id_municipio_resi = m.id_municipio
JOIN tb_uf u                       ON m.id_uf = u.id_uf
GROUP BY u.sg_uf, n.nu_ano, c.sem_pri
ORDER BY u.sg_uf, n.nu_ano, c.sem_pri;
"""
df_uf_sem = query_df(SQL_UF_SEMANAL)
print(f'Registros carregados: {len(df_uf_sem)}')

In [ ]:
# ── Loop: treina Prophet para cada UF do top 5 ──
resultados_uf = {}

for uf in top5_ufs:
    print(f'\n{"="*50}')
    print(f'Treinando Prophet para: {uf}')
    print(f'{"="*50}')

    subset = df_uf_sem[df_uf_sem['sg_uf'] == uf].copy()
    subset['ds'] = pd.to_datetime(
        subset['nu_ano'].astype(str) + '-W' +
        subset['sem_pri'].astype(str).str.zfill(2) + '-1',
        format='%G-W%V-%u', errors='coerce'
    )
    subset = subset.dropna(subset=['ds'])
    subset = subset.rename(columns={'total_casos': 'y'})[['ds', 'y']]
    subset = subset.sort_values('ds').reset_index(drop=True)

    if len(subset) < 52:
        print(f'  ⚠️ Dados insuficientes ({len(subset)} semanas). Pulando.')
        continue

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        interval_width=0.95
    )
    m.fit(subset)

    fut = m.make_future_dataframe(periods=52, freq='W')
    prev = m.predict(fut)

    # Salva resultado
    resultados_uf[uf] = {'modelo': m, 'previsao': prev, 'dados': subset}

    # Plot
    fig = m.plot(prev)
    fig.set_size_inches(14, 5)
    plt.title(f'Previsão de Casos Zika — {uf} (52 semanas à frente)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'prophet_{uf}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  ✅ Gráfico salvo: prophet_{uf}.png')

print(f'\n🏁 Prophet finalizado para {len(resultados_uf)} UFs.')

---
## 🎯 Etapa 4 — Agrupamento de Municípios (K-Means)

Objetivo: agrupar municípios brasileiros por **perfil epidemiológico** semelhante,
 identificando padrões como:
- Municípios de **alta incidência e letalidade**
- Municípios com **possível subnotificação** (muitas notificações, poucos confirmados)
- Municípios de **baixo risco**

Algoritmo: **K-Means** (partitional clustering) com seleção de K via
 Método do Cotovelo + Silhouette Score.

### 4.1 — Feature Engineering por município

In [ ]:
SQL_FEATURES = """
SELECT
    g.id_municipio_resi,
    m.nm_municipio,
    u.sg_uf,

    -- Volume
    COUNT(*) AS total_notificacoes,
    COUNT(*) FILTER (WHERE c.classi_fin = 1) AS confirmados,
    COUNT(*) FILTER (WHERE c.evolucao = 2)   AS obitos,

    -- Taxas
    ROUND(COUNT(*) FILTER (WHERE c.classi_fin = 1) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS taxa_confirmacao_pct,
    ROUND(COUNT(*) FILTER (WHERE c.evolucao = 2 AND c.classi_fin = 1) * 100.0
          / NULLIF(COUNT(*) FILTER (WHERE c.classi_fin = 1), 0), 2) AS taxa_letalidade_pct,

    -- Demografia
    ROUND(AVG(p.idade_anos)::numeric, 1) AS media_idade,
    ROUND(COUNT(*) FILTER (WHERE p.cs_sexo = 'F') * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_feminino,
    ROUND(COUNT(*) FILTER (WHERE p.cs_gestant IN (1,2,3)) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_gestantes

FROM tb_notificacao n
JOIN tb_dados_clinicos c           ON n.id_notificacao = c.id_notificacao
JOIN tb_paciente p                 ON n.id_paciente = p.id_paciente
JOIN tb_geografia_epidemiologica g ON n.id_notificacao = g.id_notificacao
JOIN tb_municipio m                ON g.id_municipio_resi = m.id_municipio
JOIN tb_uf u                       ON m.id_uf = u.id_uf
GROUP BY g.id_municipio_resi, m.nm_municipio, u.sg_uf
HAVING COUNT(*) >= 10
ORDER BY total_notificacoes DESC;
"""

df_mun = query_df(SQL_FEATURES)

print(f'Municípios com ≥10 notificações: {len(df_mun)}')
print(f'\nEstatísticas descritivas:')
df_mun.describe().round(2)

### 4.2 — Pré-processamento e normalização

In [ ]:
from sklearn.preprocessing import StandardScaler

# Features numéricas para o modelo
FEATURES = [
    'total_notificacoes', 'confirmados', 'obitos',
    'taxa_confirmacao_pct', 'taxa_letalidade_pct',
    'media_idade', 'pct_feminino', 'pct_gestantes'
]

X = df_mun[FEATURES].fillna(0)

# Normalização (K-Means é sensível à escala)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Shape da matriz de features: {X_scaled.shape}')
print(f'Features: {FEATURES}')

### 4.3 — Seleção de K: Método do Cotovelo + Silhouette Score

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))
    print(f'  K={k:2d} | Inércia={km.inertia_:,.0f} | Silhouette={sil_scores[-1]:.4f}')

# ── Gráfico duplo ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, inertias, 'bo-', linewidth=2)
ax1.set_title('Método do Cotovelo (Elbow)')
ax1.set_xlabel('K (nº de clusters)')
ax1.set_ylabel('Inércia')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, sil_scores, 'ro-', linewidth=2)
ax2.set_title('Silhouette Score por K')
ax2.set_xlabel('K (nº de clusters)')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

melhor_k = list(K_range)[np.argmax(sil_scores)]
print(f'\n🏆 Melhor K sugerido pelo Silhouette: {melhor_k}')
print('✅ Gráfico salvo: elbow_silhouette.png')

### 4.4 — Treinamento final e atribuição de clusters

In [ ]:
# Treina com o K ideal
km_final = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
df_mun['cluster'] = km_final.fit_predict(X_scaled)

print(f'Distribuição dos clusters:')
print(df_mun['cluster'].value_counts().sort_index())

### 4.5 — Perfil médio dos clusters (interpretação)

In [ ]:
# ── Tabela de perfil ──
perfil = df_mun.groupby('cluster')[FEATURES].mean().round(2)
perfil['qtd_municipios'] = df_mun.groupby('cluster').size().values

print('=== Perfil Médio dos Clusters ===')
print(perfil.to_string())

# ── Heatmap do perfil ──
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    perfil[FEATURES].T, annot=True, fmt='.1f',
    cmap='YlGnBu', linewidths=0.5, ax=ax
)
ax.set_title('Perfil Médio dos Clusters de Municípios')
ax.set_xlabel('Cluster')
ax.set_ylabel('Feature')

plt.tight_layout()
plt.savefig('perfil_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: perfil_clusters.png')

### 4.6 — Visualização 2D com PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

var_explicada = pca.explained_variance_ratio_
print(f'Variância explicada: PC1={var_explicada[0]*100:.1f}% | PC2={var_explicada[1]*100:.1f}%')

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=df_mun['cluster'], cmap='Set2',
    alpha=0.7, edgecolors='k', linewidths=0.5, s=50
)

ax.set_title('Clusters de Municípios — Projeção PCA 2D')
ax.set_xlabel(f'PC1 ({var_explicada[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explicada[1]*100:.1f}%)')
plt.colorbar(scatter, label='Cluster')

plt.tight_layout()
plt.savefig('clusters_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: clusters_pca.png')

### 4.7 — Top 10 municípios por cluster

In [ ]:
# Exibe os 10 municípios com mais notificações em cada cluster
for c in sorted(df_mun['cluster'].unique()):
    print(f'\n{"="*60}')
    print(f'CLUSTER {c}')
    print(f'{"="*60}')
    top = (df_mun[df_mun['cluster'] == c]
           .nlargest(10, 'total_notificacoes')
           [['nm_municipio', 'sg_uf', 'total_notificacoes',
             'confirmados', 'taxa_confirmacao_pct', 'taxa_letalidade_pct']])
    print(top.to_string(index=False))

---
## 📝 Etapa 5 — Conclusões e Insights Epidemiológicos

*(Preencher após execução do notebook com base nos resultados obtidos)*

### Sazonalidade
- [ ] Confirmar se os picos coincidem com o período chuvoso (dez–mar)
- [ ] Documentar a intensidade do componente sazonal

### Tendência por UF
- [ ] Listar as UFs com tendência de alta/queda
- [ ] Identificar UFs com surtos atípicos

### Previsão (Prophet)
- [ ] Reportar RMSE, MAE e MAPE do modelo nacional
- [ ] Interpretar a previsão: espera-se aumento ou queda no próximo ano?
- [ ] Comparar desempenho entre UFs

### Clustering (K-Means)
- [ ] Descrever cada cluster em linguagem epidemiológica
- [ ] Identificar municípios que merecem atenção especial da vigilância
- [ ] Reportar o Silhouette Score do modelo final

### Métricas Consolidadas

| Modelo | Métrica | Valor |
|--------|---------|-------|
| Prophet (nacional) | MAPE | *preencher* |
| Prophet (nacional) | RMSE | *preencher* |
| K-Means | Silhouette Score | *preencher* |
| K-Means | K escolhido | *preencher* |

---
*Notebook gerado automaticamente para o projeto Zika Vírus — SINAN 2018–2026*  
*Queries alinhadas ao schema: `01_schema.sql` | Views: `03_views.sql`*